# 09a. 가설 검증 Deep Dive

## 목적
08편 LLM 분류 결과(neg_reviews_classified)를 활용하여 6개 가설을 검증한다.

## 가설 목록
| # | 가설 | 데이터 |
|---|------|--------|
| 1 | 메인스토리 종료(20h 전후)에서 감성 분기 | 플레이타임 구간별 카테고리 |
| 2 | 라이트 유저(2-10h)의 높은 긍정률에는 이유가 있다 | regular 구간 긍정 키워드 (→ 09b에서 심화) |
| 3 | 일본 유저의 낮은 긍정률 (→ 09c에서 심화) | 여기서는 기초 통계만 |
| 4 | 부정 리뷰는 하나가 아니다 — 유형 분리 | tone + actual_sentiment 조합 |
| 5 | 개발자 응답 패턴 — 어디에 반응하고, 어디를 놓쳤는가 | has_dev_response × category |
| 6 | 2시간 미만 환불 미실행 유저 | casual + steam_purchase |

## 분석 기준 method
rules와 anthropic 두 가지가 있으므로, **anthropic을 기본 분석**에 사용하고 rules는 비교용으로 참조.

---
## 1. 라이브러리 & 데이터 로드

In [2]:
import sys, os, sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import platform
import warnings

warnings.filterwarnings('ignore')

if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.append('..')
from config import DB_PATH, EVENTS

db_path = os.path.join('..', DB_PATH)
conn = sqlite3.connect(db_path)

figures_dir = os.path.join('..', 'reports', 'figures')
os.makedirs(figures_dir, exist_ok=True)

# 분석 기준 method
METHOD = 'anthropic'

# 분류 결과 로드
cls = pd.read_sql(f"""
    SELECT * FROM neg_reviews_classified
    WHERE classify_method = '{METHOD}'
""", conn)

cls_en = cls[cls['language_group'] == 'english'].copy()
cls_ko = cls[cls['language_group'] == 'korean'].copy()

# 전체 리뷰 (긍정 포함) — 비율 계산용
all_reviews = pd.read_sql("SELECT * FROM reviews", conn)

print(f'분석 method: {METHOD}')
print(f'부정 영어: {len(cls_en):,}건, 한국어: {len(cls_ko):,}건')
print(f'전체 리뷰: {len(all_reviews):,}건')

분석 method: anthropic
부정 영어: 1,649건, 한국어: 298건
전체 리뷰: 145,877건


In [3]:
CATEGORIES = ['gameplay', 'story_content', 'repetition', 'technical', 'company', 'forced_neg', 'other']

colors = {
    'gameplay': '#FF6B6B', 'story_content': '#4ECDC4', 'repetition': '#45B7D1',
    'technical': '#96CEB4', 'company': '#FFEAA7', 'forced_neg': '#DDA0DD', 'other': '#C0C0C0'
}

segment_order = ['casual(<2h)', 'regular(2-10h)', 'engaged(10-30h)', 'engaged(30-50h)', 'hardcore(50h+)']

---
## 가설 1: 메인스토리 종료(20h 전후)에서 감성 분기

> 10~30시간 구간의 긍정률이 소폭 하락하는 패턴을 확인했다.
> 20시간 이후 부정 리뷰는 "콘텐츠가 끝났다"는 허탈감에서 비롯됐을 가능성이 크다.
> 30~50시간, 50시간 이상을 플레이하고도 부정 평가를 남긴 유저들이 특히 궁금하다.

In [4]:
# ── 5시간 단위 세분화 (0~60h+) ──
def fine_band(hours):
    if pd.isna(hours):
        return 'unknown'
    if hours < 2:
        return '00-02h'
    elif hours < 5:
        return '02-05h'
    elif hours < 10:
        return '05-10h'
    elif hours < 15:
        return '10-15h'
    elif hours < 20:
        return '15-20h'
    elif hours < 25:
        return '20-25h'
    elif hours < 30:
        return '25-30h'
    elif hours < 40:
        return '30-40h'
    elif hours < 50:
        return '40-50h'
    else:
        return '50h+'

band_order = ['00-02h','02-05h','05-10h','10-15h','15-20h',
              '20-25h','25-30h','30-40h','40-50h','50h+']

cls_en['fine_band'] = cls_en['playtime_hours'].apply(fine_band)

# ── 구간별 카테고리 분포 ──
ct = pd.crosstab(cls_en['fine_band'], cls_en['category'], normalize='index') * 100
ct = ct.reindex(index=band_order, columns=CATEGORIES).fillna(0)

# 구간별 건수
band_counts = cls_en['fine_band'].value_counts().reindex(band_order).fillna(0).astype(int)

print('=== 5시간 단위 세분화: 부정 카테고리 구성 (영어) ===')
for band in band_order:
    n = band_counts.get(band, 0)
    print(f'\n{band} ({n}건):')
    if n == 0:
        continue
    for cat in CATEGORIES:
        pct = ct.loc[band, cat] if band in ct.index else 0
        if pct > 3:
            bar = '█' * int(pct / 2)
            print(f'  {cat:15s} {pct:>5.1f}% {bar}')

=== 5시간 단위 세분화: 부정 카테고리 구성 (영어) ===

00-02h (364건):
  gameplay         27.7% █████████████
  story_content     6.9% ███
  repetition       14.3% ███████
  technical        20.1% ██████████
  company           8.8% ████
  other            20.3% ██████████

02-05h (166건):
  gameplay         25.9% ████████████
  story_content     5.4% ██
  repetition       31.3% ███████████████
  technical        13.3% ██████
  company           7.2% ███
  forced_neg        3.0% █
  other            13.9% ██████

05-10h (227건):
  gameplay         23.8% ███████████
  story_content    11.0% █████
  repetition       39.2% ███████████████████
  technical         7.0% ███
  company           8.4% ████
  other             8.8% ████

10-15h (187건):
  gameplay         32.6% ████████████████
  story_content    18.2% █████████
  repetition       25.1% ████████████
  technical         3.7% █
  company          10.7% █████
  other             8.0% ████

15-20h (149건):
  gameplay         33.6% ████████████████
  story

In [5]:
# ── 스택드 바 차트: 5시간 단위 ──
fig = go.Figure()
for cat in CATEGORIES:
    fig.add_trace(go.Bar(
        name=cat, x=band_order,
        y=[ct.loc[b, cat] if b in ct.index else 0 for b in band_order],
        marker_color=colors.get(cat, '#999')))

fig.update_layout(
    barmode='stack',
    title='가설1: 플레이타임 5시간 단위별 부정 카테고리 구성 (영어)',
    xaxis_title='플레이타임', yaxis_title='비중 (%)',
    height=500, width=1000, template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02))

# 20시간 기준선
fig.add_vline(x=4.5, line_dash='dash', line_color='red',
              annotation_text='메인스토리 종료 추정', annotation_position='top left')
fig.show()

In [6]:
# ── 핵심: repetition 비중이 20h 이후 급증하는지 ──
rep_trend = ct['repetition'].reindex(band_order).fillna(0)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=band_order, y=rep_trend.values,
    mode='lines+markers+text',
    text=[f'{v:.0f}%' for v in rep_trend.values],
    textposition='top center',
    line=dict(color='#45B7D1', width=3),
    marker=dict(size=10)))

fig2.add_vline(x=4.5, line_dash='dash', line_color='red',
               annotation_text='20h', annotation_position='top left')

fig2.update_layout(
    title='가설1: "repetition" 비중의 플레이타임별 변화',
    xaxis_title='플레이타임', yaxis_title='repetition 비중 (%)',
    height=400, width=900, template='plotly_white')
fig2.show()

In [7]:
# ── 50시간+ 부정 유저: 무엇이 이들을 부정으로 돌렸나? ──
hardcore = cls_en[cls_en['fine_band'] == '50h+'].copy()
print(f'=== 50시간+ 부정 유저 ({len(hardcore)}건) ===')
print(f'평균 플레이타임: {hardcore["playtime_hours"].mean():.0f}시간')
print(f'\n카테고리:')
print(hardcore['category'].value_counts().to_string())
print(f'\ntone:')
print(hardcore['tone'].value_counts().to_string())
print(f'\nactual_sentiment:')
print(hardcore['actual_sentiment'].value_counts().to_string())

# constructive한 50h+ 리뷰 (가장 가치 있는 피드백)
constructive_hc = hardcore[hardcore['tone'] == 'constructive']
print(f'\n--- 건설적 비판 (50h+, {len(constructive_hc)}건) ---')
for _, row in constructive_hc.head(5).iterrows():
    print(f'[{row["playtime_hours"]:.0f}h] [{row["category"]}]')
    print(f'  {row["review_text"][:200]}')
    if row.get('reason'):
        print(f'  → {row["reason"]}')
    print()

=== 50시간+ 부정 유저 (140건) ===
평균 플레이타임: 85시간

카테고리:
category
gameplay         43
repetition       33
company          28
technical        12
other             9
story_content     9
forced_neg        6

tone:
tone
constructive    81
emotional       28
mixed           26
troll            5

actual_sentiment:
actual_sentiment
mixed_negative                90
pure_negative                 36
positive_but_negative_vote    11
joke                           3

--- 건설적 비판 (50h+, 81건) ---
[323h] [technical]
  To be clear, I absolutely adore this game, as is evident given the hours I've sunk into it.  Unfortunately I am now locked out from being able to play.  I have an old computer that I spent a stupid am
  → Technical issue with macOS compatibility preventing play of a game they love.

[70h] [repetition]
  The Positives: The start of the game is really solid and has a really tight, satisfying gameplay loop where you dive down, collect fish, sell your fish in your restaurant, and then use those f

### 가설 1 결론

*(실행 후 결과를 보고 여기에 작성)*

- 20시간 이후 `repetition` 비중이 __% → __% 로 변화 (증가/감소/유지)
- 50시간+ 유저의 주요 불만: ___
- **인사이트**: ___
- **제언**: ___

---
## 가설 2: 라이트 유저(2-10h)의 높은 긍정률

> 2~10시간은 전체 콘텐츠를 경험하기엔 짧은 시간임에도 긍정률이 매우 높다.
> 여기서는 부정 리뷰 측면에서 분석하고, 긍정 리뷰 분류는 09b에서 진행한다.

### 접근: regular(2-10h)의 부정 유저는 왜 부정인가?

In [8]:
regular_neg = cls_en[cls_en['play_segment'] == 'regular(2-10h)'].copy()

print(f'=== regular(2-10h) 부정 유저 ({len(regular_neg)}건) ===')
print(f'\n카테고리:')
print(regular_neg['category'].value_counts().to_string())
print(f'\ntone:')
print(regular_neg['tone'].value_counts().to_string())
print(f'\nactual_sentiment:')
print(regular_neg['actual_sentiment'].value_counts().to_string())

=== regular(2-10h) 부정 유저 (390건) ===

카테고리:
category
repetition       141
gameplay          96
other             42
technical         37
story_content     34
company           31
forced_neg         9

tone:
tone
constructive    243
emotional        98
mixed            37
troll            12

actual_sentiment:
actual_sentiment
mixed_negative                185
pure_negative                 177
positive_but_negative_vote     18
joke                           10


In [9]:
# ── regular vs 다른 구간의 카테고리 구성 차이 ──
ct_seg = pd.crosstab(cls_en['play_segment'], cls_en['category'], normalize='index') * 100
ct_seg = ct_seg.reindex(index=segment_order, columns=CATEGORIES).fillna(0)

print('=== 구간별 카테고리 비중 차이 (regular 기준) ===')
if 'regular(2-10h)' in ct_seg.index:
    for cat in CATEGORIES:
        reg_pct = ct_seg.loc['regular(2-10h)', cat]
        for seg in segment_order:
            if seg == 'regular(2-10h)' or seg not in ct_seg.index:
                continue
            diff = ct_seg.loc[seg, cat] - reg_pct
            if abs(diff) >= 5:
                print(f'  {cat:15s}: regular {reg_pct:.1f}% → {seg} {ct_seg.loc[seg, cat]:.1f}% ({diff:+.1f}pp)')

=== 구간별 카테고리 비중 차이 (regular 기준) ===
  gameplay       : regular 24.6% → engaged(10-30h) 34.4% (+9.8pp)
  gameplay       : regular 24.6% → engaged(30-50h) 38.3% (+13.7pp)
  gameplay       : regular 24.6% → hardcore(50h+) 30.7% (+6.1pp)
  story_content  : regular 8.7% → engaged(10-30h) 13.8% (+5.1pp)
  repetition     : regular 36.2% → casual(<2h) 14.2% (-22.0pp)
  repetition     : regular 36.2% → engaged(10-30h) 26.6% (-9.6pp)
  repetition     : regular 36.2% → engaged(30-50h) 25.5% (-10.6pp)
  repetition     : regular 36.2% → hardcore(50h+) 23.6% (-12.6pp)
  technical      : regular 9.5% → casual(<2h) 20.2% (+10.7pp)
  company        : regular 7.9% → hardcore(50h+) 20.0% (+12.1pp)
  other          : regular 10.8% → casual(<2h) 20.4% (+9.7pp)


In [10]:
# ── regular 구간에서 "긍정인데 비추천" 유저 ──
pos_but_neg = regular_neg[regular_neg['actual_sentiment'] == 'positive_but_negative_vote']
print(f'\n=== regular에서 긍정인데 비추천 ({len(pos_but_neg)}건) ===')
for _, row in pos_but_neg.iterrows():
    print(f'[{row["category"]}] {row["review_text"][:200]}')
    print(f'  → reason: {row.get("reason", "")}')
    print()


=== regular에서 긍정인데 비추천 (18건) ===
[other] I think this game is extremely well made, polished as heck, has tight controls, a good story, well thought out gameplay loops and mechanics... but it isn't for me at all.

If "iPad kid" was turned int
  → reason: Reviewer explicitly praises the game's quality but states it's not their personal preference.

[company] The game is good. However, the limited time DLC has changed my thoughts towards this game. You can only legitimately buy it for a short time, but it'll be downloadable elsewhere for years. I thought G
  → reason: States game is good but limited time DLC practice changed their opinion, showing FOMO has become company pattern.

[company] I like the game a lot but can not honestly support a game that uses time limited DLC in order to get people to FOMO into spending more. Also friendly reminder that this game was not made by indie devs
  → reason: Explicitly states liking the game but cannot support due to FOMO DLC practices and Nexon 

# 긍정적 리뷰 + 비추천(부정 평가) 데이터 분석 (총 18건)

게임 자체의 완성도나 재미는 인정하면서도, **외부적 요인(회사 정책), 기술적 결함, 접근성 문제, 혹은 고의적인 평점 테러**를 이유로 비추천을 남긴 특이 케이스.

---

##  Company (회사 정책 및 BM 비판) - 7건
게임을 호평하면서도 퍼블리셔(Nexon)에 대한 반감이나 기간 한정 DLC 상술(FOMO)에 대한 보이콧 목적으로 비추천을 누른 그룹.

* **리뷰 1:** "The game is good. However, the limited time DLC has changed my thoughts..."
  * **분석:** 게임은 좋으나, 기간 한정 DLC로 FOMO(소외 불안)를 유발하는 상술에 실망함.
* **리뷰 2:** "I like the game a lot but can not honestly support a game that uses time limited DLC... not made by indie devs"
  * **분석:** 넥슨이라는 대기업이 인디인 척하며 FOMO 마케팅을 하는 것을 지지할 수 없음.
* **리뷰 3:** "I really enjoyed this game... However, I like to take my time wit[h]"
  * **분석:** 게임은 즐겼으나, 기간 한정 DLC가 자신의 여유로운 플레이 성향(수집 요소)을 방해함.
* **리뷰 4:** "Just going to get it out of the way, the game is great. This time limited DLC BS is not..."
  * **분석:** 게임은 훌륭하지만 기간 한정 DLC 정책을 비판. 복돌이(해적판) 유저가 오히려 풀 컨텐츠를 즐기는 역차별 지적.
* **리뷰 5:** "Great game from a company I hate... Nexon blows, enough said."
  * **분석:** 훌륭한 게임이지만 퍼블리셔인 넥슨을 혐오하기 때문에 비추천함.
* **리뷰 6:** "It's a shame. The game is fun enough but this has Nexon stain on it."
  * **분석:** 게임은 재미있지만 '넥슨의 얼룩'이 묻어 있어 추천할 수 없음.
* **리뷰 7:** "It's a very good game... Thumbs down to get attention from devs..."
  * **분석:** 매우 좋은 게임이지만, 개발자의 관심을 끌기 위해 고의로 비추천함. (가격 책정 및 반복성 지적)

---

##  Technical (기술적 결함) - 4건
게임은 마음에 들지만, 본인의 하드웨어 환경에서 정상적인 플레이가 불가능해 비추천을 남긴 그룹입니다.

* **리뷰 8:** "I love this game... BUT it's unplayable on my steam deck. It literally crashes every 2-5 minutes..."
  * **분석:** 스팀덱에서 2~5분마다 튕기는 치명적인 크래시 문제 발생.
* **리뷰 9:** "From what I've been able to play, I really enjoy this game. However... black screen..."
  * **분석:** 플레이 경험은 즐거웠으나, 컴퓨터를 재부팅해야 하는 블랙스크린 프리징 현상 발생.
* **리뷰 10:** "THIS IS A GREAT GAME! ... getting screen tare very often and there is no VSYNC option..."
  * **분석:** 훌륭한 게임이나, 심각한 화면 테어링(찢어짐) 현상이 있고 VSYNC 옵션이 부재함.
* **리뷰 11:** "This IS a good game... resolution options are extremely limited."
  * **분석:** 좋은 게임임은 확실하나, 해상도 옵션이 제한적이고 완벽한 전체화면을 지원하지 않음.

---

##  Gameplay (접근성 및 신체적 불편함) - 2건
게임성은 훌륭하지만, 특정 조작(QTE 연타)이 신체적 통증을 유발하여 접근성 측면에서 비추천한 그룹.

* **리뷰 12:** "This is a great game... If you have any sort of carpal tunnel or hand/wrist issues, DO NOT BUY THIS..."
  * **분석:** 훌륭한 게임이지만, 수근관 증후군 등 손/손목 질환이 있는 사람에게는 버튼 연타가 너무 고통스러움.
* **리뷰 13:** "I love this game, love it love it love it. But the accessibility is awful. The QTEs... give me RSI..."
  * **분석:** 게임을 매우 사랑하지만, 작살 QTE 액션이 반복사용긴장성손상증후군(RSI)을 유발할 정도로 접근성이 최악임.

---

##  Forced_neg (고의적 비추천 / 장난) - 4건
텍스트 전체가 찬양 일색이거나 유머성 글임에도 불구하고 시스템상 비추천 버튼을 누른 (혹은 평점 밸런스를 맞추려는) 그룹.

* **리뷰 14:** "Really fun game"
  * **분석:** 단순히 "정말 재밌는 게임"이라고 적어놓고 부정 평가를 누른 오류성 리뷰.
* **리뷰 15:** "This game is brilliant, the graphics, the story, the mechanics..."
  * **분석:** 그래픽, 스토리, 메커니즘 등 모든 면을 극찬했으나 부정 평가 처리됨.
* **리뷰 16:** "Just to balance out those positive reviews. The game is colourful & fun..."
  * **분석:** 게임은 재미있지만 단순히 '긍정 평가가 너무 많아 밸런스를 맞추기 위해' 비추천을 누름.
* **리뷰 17:** "This has serious health consequences. Read our first play thru! ... Two hours have passed..."
  * **분석:** 게임이 너무 몰입감 있고 중독성 있어서 시간 가는 줄 몰랐다는 호평을 '건강상의 심각한 결과'라며 유머러스하게 비꼰 리뷰.

---

## Other (개인적 취향 불일치) - 1건
객관적인 완성도는 인정하지만 단순히 본인 취향이 아니라는 이유.

* **리뷰 18:** "I think this game is extremely well made... but it isn't for me at all. If "iPad kid" was turned int[o]..."
  * **분석:** 조작감과 루프 등 완성도는 극찬하지만, 끊임없는 자극('아이패드 키즈'에 비유)이 본인 성향과는 전혀 맞지 않음.

### 가설 2 결론

*(실행 후 작성)*

- regular(2-10h) 부정의 주 카테고리: ___
- 다른 구간 대비 차이: ___
- 긍정인데 비추천 비율: ___
- **→ 09b에서 긍정 리뷰 분류 후 "어떤 요소가 보편적 매력인지" 확인 필요**

---
## 가설 3: 일본 유저의 낮은 긍정률 (기초 통계)

> 여기서는 전체 reviews 테이블에서 일본어 기초 통계만 확인.
> 일본어 텍스트 분석은 09c에서 진행.

In [11]:
# ── 언어별 긍정률 Top 15 ──
lang_stats = pd.read_sql("""
    SELECT language,
           COUNT(*) as total,
           SUM(voted_up) as positive,
           ROUND(AVG(voted_up) * 100, 1) as positive_rate,
           ROUND(AVG(playtime_at_review / 60.0), 1) as avg_hours
    FROM reviews
    GROUP BY language
    HAVING total >= 100
    ORDER BY total DESC
""", conn)

print('=== 언어별 긍정률 (100건 이상) ===')
print(lang_stats.to_string(index=False))

=== 언어별 긍정률 (100건 이상) ===
  language  total  positive  positive_rate  avg_hours
  schinese  54059     51929           96.1       25.8
   english  51732     50021           96.7       28.0
   koreana  11318     11003           97.2       26.7
  tchinese   6690      6597           98.6       32.5
 brazilian   4452      4429           99.5       28.8
   spanish   3925      3890           99.1       26.9
    german   3641      3558           97.7       27.7
    french   1989      1936           97.3       28.0
   russian   1269      1217           95.9       27.5
   turkish   1144      1113           97.3       21.8
      thai   1087      1080           99.4       30.2
  japanese   1023       938           91.7       36.6
     latam    853       847           99.3       29.9
    polish    543       534           98.3       25.0
   italian    542       534           98.5       27.6
     czech    252       251           99.6       25.2
portuguese    209       209          100.0       26.1
vi

In [12]:
# ── 일본어 vs 영어 vs 한국어 상세 비교 ──
target_langs = ['english', 'japanese', 'korean']
lang_compare = lang_stats[lang_stats['language'].isin(target_langs)].copy()

print('\n=== 영어/일본어/한국어 비교 ===')
print(lang_compare.to_string(index=False))

# ── 일본어 플레이타임 분포 ──
jp_reviews = all_reviews[all_reviews['language'] == 'japanese'].copy()
jp_reviews['hours'] = jp_reviews['playtime_at_review'] / 60.0

def segment_label(h):
    if h < 2: return 'casual(<2h)'
    elif h < 10: return 'regular(2-10h)'
    elif h < 30: return 'engaged(10-30h)'
    elif h < 50: return 'engaged(30-50h)'
    else: return 'hardcore(50h+)'

jp_reviews['segment'] = jp_reviews['hours'].apply(segment_label)

print('\n=== 일본어: 플레이타임 구간별 긍정률 ===')
for seg in segment_order:
    subset = jp_reviews[jp_reviews['segment'] == seg]
    if len(subset) == 0:
        continue
    rate = subset['voted_up'].mean() * 100
    print(f'  {seg:20s}: {len(subset):>4}건, 긍정률 {rate:.1f}%')


=== 영어/일본어/한국어 비교 ===
language  total  positive  positive_rate  avg_hours
 english  51732     50021           96.7       28.0
japanese   1023       938           91.7       36.6

=== 일본어: 플레이타임 구간별 긍정률 ===
  casual(<2h)         :   31건, 긍정률 74.2%
  regular(2-10h)      :  168건, 긍정률 91.1%
  engaged(10-30h)     :  358건, 긍정률 89.9%
  engaged(30-50h)     :  255건, 긍정률 92.9%
  hardcore(50h+)      :  211건, 긍정률 96.2%


In [13]:
# ── 언어별 긍정률 비교 (주요 언어) ──
major_langs = lang_stats[lang_stats['total'] >= 500].copy()
major_langs = major_langs.sort_values('positive_rate')

fig = go.Figure(go.Bar(
    y=major_langs['language'],
    x=major_langs['positive_rate'],
    orientation='h',
    marker_color=['#FF6B6B' if l == 'japanese' else '#45B7D1' for l in major_langs['language']],
    text=[f"{v:.1f}%" for v in major_langs['positive_rate']],
    textposition='outside'))

fig.update_layout(
    title='가설3: 언어별 긍정률 비교 (500건 이상)',
    xaxis_title='긍정률 (%)', xaxis_range=[70, 100],
    height=500, width=800, template='plotly_white')
fig.show()

### 가설 3 결론 (기초)

*(실행 후 작성)*

- 일본어 긍정률: __% (영어 대비 __pp 차이)
- 플레이타임 구간별 패턴: ___
- **→ 09c에서 일본어 텍스트 전처리 + LLM 분류 후 원인 분석**

---
## 가설 4: 부정 리뷰는 하나가 아니다 — 유형 분리

> "중립을 주고 싶었지만 어쩔 수 없이 부정을 선택한 유저",
> "게임 구조적 문제를 비판하는 유저",
> "넥슨·프라이버시 이슈로 부정을 준 유저"
> 가 모두 같은 버킷에 묶여 있다.

### 접근: tone × actual_sentiment 조합으로 부정 유형 매트릭스 생성

In [14]:
# ── tone × actual_sentiment 매트릭스 ──
matrix = pd.crosstab(cls_en['tone'], cls_en['actual_sentiment'])
matrix_pct = pd.crosstab(cls_en['tone'], cls_en['actual_sentiment'], normalize='all') * 100

print('=== 부정 리뷰 유형 매트릭스 (영어) ===')
print('\n건수:')
print(matrix.to_string())
print('\n비율 (%):')
print(matrix_pct.round(1).to_string())

=== 부정 리뷰 유형 매트릭스 (영어) ===

건수:
actual_sentiment  joke  mixed_negative  positive_but_negative_vote  pure_negative
tone                                                                             
constructive         0             661                          45            336
emotional            0              52                           8            310
mixed                0             126                          20             43
troll               40               0                           7              1

비율 (%):
actual_sentiment  joke  mixed_negative  positive_but_negative_vote  pure_negative
tone                                                                             
constructive       0.0            40.1                         2.7           20.4
emotional          0.0             3.2                         0.5           18.8
mixed              0.0             7.6                         1.2            2.6
troll              2.4             0.0                   

In [15]:
# ── 부정 유형 분류 (비즈니스 관점) ──
def classify_neg_type(row):
    """부정 리뷰를 비즈니스 관점으로 재분류"""
    tone = row.get('tone', 'mixed')
    sentiment = row.get('actual_sentiment', 'pure_negative')
    cat = row.get('category', 'other')

    if sentiment in ('positive_but_negative_vote', 'joke'):
        return '가짜 부정 (실제 긍정/장난)'
    elif cat == 'company':
        return '게임 외부 불만 (회사/가격)'
    elif tone == 'constructive':
        return '건설적 비판 (개선 가능)'
    elif tone == 'emotional':
        return '감정적 분풀이'
    elif sentiment == 'mixed_negative':
        return '아쉬움 (긍정+부정 혼재)'
    else:
        return '기타 부정'

cls_en['neg_type'] = cls_en.apply(classify_neg_type, axis=1)

neg_type_counts = cls_en['neg_type'].value_counts()
neg_type_pct = cls_en['neg_type'].value_counts(normalize=True) * 100

print('=== 부정 유형 분류 (비즈니스 관점) ===')
for ntype in neg_type_counts.index:
    n = neg_type_counts[ntype]
    pct = neg_type_pct[ntype]
    bar = '█' * int(pct / 2)
    print(f'  {ntype:25s} {n:>5}건 ({pct:>5.1f}%) {bar}')

=== 부정 유형 분류 (비즈니스 관점) ===
  건설적 비판 (개선 가능)              956건 ( 58.0%) ████████████████████████████
  감정적 분풀이                     305건 ( 18.5%) █████████
  가짜 부정 (실제 긍정/장난)            120건 (  7.3%) ███
  게임 외부 불만 (회사/가격)            115건 (  7.0%) ███
  아쉬움 (긍정+부정 혼재)              114건 (  6.9%) ███
  기타 부정                        39건 (  2.4%) █


In [16]:
type_colors = {
    '가짜 부정 (실제 긍정/장난)': '#DDA0DD',
    '게임 외부 불만 (회사/가격)': '#FFEAA7',
    '건설적 비판 (개선 가능)': '#4ECDC4',
    '감정적 분풀이': '#FF6B6B',
    '아쉬움 (긍정+부정 혼재)': '#45B7D1',
    '기타 부정': '#C0C0C0',
}

fig = go.Figure(go.Pie(
    labels=neg_type_counts.index.tolist(),
    values=neg_type_counts.values.tolist(),
    marker_colors=[type_colors.get(t, '#999') for t in neg_type_counts.index],
    textinfo='label+percent',
    hole=0.4))

fig.update_layout(
    title='가설4: 부정 리뷰 유형 분류 (비즈니스 관점)',
    height=500, width=700, template='plotly_white',
    annotations=[dict(text=f'{len(cls_en):,}건', x=0.5, y=0.5,
                      font_size=20, showarrow=False)])
fig.show()

In [17]:
# ── 핵심 인사이트: "실제로 개선 가능한" 부정 비율 ──
actionable = cls_en[cls_en['neg_type'] == '건설적 비판 (개선 가능)']
non_actionable = cls_en[cls_en['neg_type'].isin(['가짜 부정 (실제 긍정/장난)', '게임 외부 불만 (회사/가격)'])]

print(f'=== 액션 가능성 분석 ===')
print(f'건설적 비판 (개선 가능):          {len(actionable):>5}건 ({len(actionable)/len(cls_en)*100:.1f}%)')
print(f'게임 외부/가짜 부정 (개선 불가):  {len(non_actionable):>5}건 ({len(non_actionable)/len(cls_en)*100:.1f}%)')
print(f'\n→ 부정 리뷰의 {len(non_actionable)/len(cls_en)*100:.0f}%는 게임 개선으로 해결할 수 없는 불만')
print(f'→ 실제 게임 개선으로 잡을 수 있는 부정 리뷰: 약 {len(actionable)/len(cls_en)*100:.0f}%')

print(f'\n--- 건설적 비판의 카테고리 구성 ---')
print(actionable['category'].value_counts().to_string())

=== 액션 가능성 분석 ===
건설적 비판 (개선 가능):            956건 (58.0%)
게임 외부/가짜 부정 (개선 불가):    235건 (14.3%)

→ 부정 리뷰의 14%는 게임 개선으로 해결할 수 없는 불만
→ 실제 게임 개선으로 잡을 수 있는 부정 리뷰: 약 58%

--- 건설적 비판의 카테고리 구성 ---
category
gameplay         344
repetition       315
story_content    113
technical        112
other             71
forced_neg         1


### 가설 4 결론

*(실행 후 작성)*

- 가짜 부정(forced_neg + positive_but_neg_vote): __%
- 게임 외부 불만(company): __%
- **건설적 비판 비율: __%** → 이것만이 실제 개선 대상
- 건설적 비판의 주 카테고리: ___
- **제언**: 부정률 X%를 낮추려면 전체가 아닌 건설적 비판 Y%에 집중해야 한다

---
## 가설 5: 개발자 응답 패턴

> 개발자 응답이 달린 리뷰의 긍정률이 23%로 낮다 → 주로 부정 리뷰에 대응.
> 추천 수가 높고 공감 많지만 개발자 응답이 없는 리뷰는?

In [18]:
# ── 카테고리별 개발자 응답률 ──
dev_pattern = cls_en.groupby('category').agg(
    total=('review_id', 'count'),
    with_response=('has_dev_response', 'sum'),
).reset_index()
dev_pattern['response_rate'] = (dev_pattern['with_response'] / dev_pattern['total'] * 100).round(1)
dev_pattern = dev_pattern.sort_values('response_rate', ascending=False)

print('=== 카테고리별 개발자 응답률 ===')
print(dev_pattern.to_string(index=False))

=== 카테고리별 개발자 응답률 ===
     category  total  with_response  response_rate
    technical    169             68           40.2
     gameplay    507            114           22.5
   repetition    424             59           13.9
story_content    165             22           13.3
        other    183             14            7.7
   forced_neg     42              2            4.8
      company    159              6            3.8


In [19]:
# ── 영향력 크지만 개발자 응답 없는 리뷰 ──
missed = cls_en[
    (cls_en['has_dev_response'] == 0) &
    (cls_en['weighted_vote_score'] > 0.4) &
    (cls_en['tone'] == 'constructive')
].sort_values('weighted_vote_score', ascending=False)

print(f'=== 영향력 높은데 응답 없는 건설적 리뷰 (Top 10) ===')
for _, row in missed.head(10).iterrows():
    print(f'[score={row["weighted_vote_score"]:.3f}] [votes_up={row["votes_up"]}] [{row["category"]}]')
    print(f'  {row["review_text"][:200]}')
    print()

=== 영향력 높은데 응답 없는 건설적 리뷰 (Top 10) ===
[score=0.793] [votes_up=123] [repetition]
  Okay let's do the opposite of what the game does here and get straight to the point.

Relaxing and enjoyable at the start, the game initially sets a great tone.
However as you progress it gets increas

[score=0.781] [votes_up=86] [repetition]
  I wish there was a neutral vote option in this case, since I normally don't down vote games in a review.

To start off with, the characters and atmosphere is great! Super high-quality, and you can tel

[score=0.775] [votes_up=154] [technical]
  Dave the Diver takes us on a journey in the sea of data.

[h1]What Does This Product Do?[/h1]
A little more than usual. The product was realized with the help of the Unity engine, has corresponding an

[score=0.772] [votes_up=50] [repetition]
  TL;DR - game has TOO much and it becomes tedious/a chore to manage everything.

This game is beautiful, quirky, and varied. I was super excited for the game since it combined two thin

In [20]:
# ── 개발자 응답 있는 vs 없는: 카테고리 구성 차이 ──
with_resp = cls_en[cls_en['has_dev_response'] == 1]
no_resp = cls_en[cls_en['has_dev_response'] == 0]

if len(with_resp) > 0:
    comp = pd.DataFrame({
        '응답O (%)': with_resp['category'].value_counts(normalize=True).reindex(CATEGORIES, fill_value=0) * 100,
        '응답X (%)': no_resp['category'].value_counts(normalize=True).reindex(CATEGORIES, fill_value=0) * 100,
    }).round(1)
    comp['차이(pp)'] = (comp['응답O (%)'] - comp['응답X (%)']).round(1)
    print('=== 개발자 응답 유무별 카테고리 구성 ===')
    print(comp.to_string())
else:
    print('개발자 응답이 있는 부정 리뷰가 없습니다.')

=== 개발자 응답 유무별 카테고리 구성 ===
               응답O (%)  응답X (%)  차이(pp)
category                               
gameplay          40.0     28.8    11.2
story_content      7.7     10.5    -2.8
repetition        20.7     26.8    -6.1
technical         23.9      7.4    16.5
company            2.1     11.2    -9.1
forced_neg         0.7      2.9    -2.2
other              4.9     12.4    -7.5


### 가설 5 결론

*(실행 후 작성)*

- 개발자가 가장 많이 응답한 카테고리: ___
- 개발자가 놓친 카테고리: ___
- 영향력 높은데 응답 없는 리뷰 패턴: ___
- **제언**: ___

---
## 종합 결론 & 다음 단계

*(6개 가설 검증 후 종합 정리)*

### 핵심 발견
1. ...
2. ...

### 비즈니스 제언
- 기획자용: ...
- 사업팀용: ...

### 다음 노트북
- **09b**: 긍정 리뷰 층화 샘플링 + Sonnet 분류 → 가설2 심화
- **09c**: 일본어 전처리(spaCy ja) + Sonnet 분류 → 가설3 심화

In [23]:
conn.close()
print('DB 연결 종료')

DB 연결 종료
